<a href="https://colab.research.google.com/github/sumerahmughal/iat360-vegetable-recognition/blob/main/notebooks/Sumerah_LearningRate_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Person A – # Learning Rate Experiments + Evaluation on Our Own Photos
Baseline (from `Vegetable_Classification_Baseline.ipynb`):

This notebook tests three learning rates (0.001, 0.005, 0.02)
using YOLO11n-cls, with 15 epochs, image size 224,
batch size 16, and seed 42.

Each model is evaluated on the Kaggle test set and
our group's 120 original photographs.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

Mounted at /content/drive


## Settings – check these two paths

In [2]:
PROJECT = "/content/drive/MyDrive/IAT 360 CV PROJECT"        # same folder the baseline used
ZIP_PATH = f"{PROJECT}/vegetable_dataset.zip"
OWN_PHOTOS_ROOT = PROJECT   # folder that contains everyone's photo folders (Alyssa, etc.). Searched recursively.
RESULTS = f"{PROJECT}/training_results"
BASELINE_WEIGHTS = f"{RESULTS}/baseline/weights/best.pt"

CLASSES = ["Broccoli", "Capsicum", "Carrot", "Potato", "Tomato"]
BASE = dict(epochs=15, imgsz=224, seed=42)
LEARNING_RATES = [0.001, 0.005, 0.02]
OPTIMIZER = "AdamW"

## 1. Unzip the online dataset

In [3]:
import zipfile, os
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall("/content/")
DATA = "/content/vegetable_dataset"
for s in ["train","val","test"]:
    print(s, {c: len(os.listdir(f"{DATA}/{s}/{c}")) for c in sorted(os.listdir(f"{DATA}/{s}")) if os.path.isdir(f"{DATA}/{s}/{c}")})

train {'Broccoli': 1000, 'Capsicum': 1000, 'Carrot': 1000, 'Potato': 1000, 'Tomato': 1000}
val {'Broccoli': 200, 'Capsicum': 200, 'Carrot': 200, 'Potato': 200, 'Tomato': 200}
test {'Broccoli': 200, 'Capsicum': 200, 'Carrot': 200, 'Potato': 200, 'Tomato': 200}


## 2. Build the own-photo test set
Finds every photo named like `carrot_alyssa_003.jpeg` anywhere under `OWN_PHOTOS_ROOT` and sorts it into a class folder by the first word of its name.
Check the printed counts: you want ~24 per class (6 × 4 people) and 120 total. Anything listed as *skipped* has a name that didn't match – rename it in Drive and rerun.

In [4]:
import re, shutil, glob
from collections import Counter
OWN = "/content/own_test_ds"
shutil.rmtree(OWN, ignore_errors=True)
for c in CLASSES: os.makedirs(f"{OWN}/test/{c}", exist_ok=True)
os.symlink(f"{DATA}/train", f"{OWN}/train"); os.symlink(f"{DATA}/val", f"{OWN}/val")   # ultralytics wants these to exist

lookup = {c.lower(): c for c in CLASSES}
lookup.update({"pepper": "Capsicum", "bellpepper": "Capsicum", "tomatoes": "Tomato", "carrots": "Carrot", "potatoes": "Potato"})
pat = re.compile(r"^([a-z]+)_([a-z]+)_(\d+)", re.I)
counts, people, skipped, seen = Counter(), Counter(), [], set()
for p in glob.glob(f"{OWN_PHOTOS_ROOT}/**/*", recursive=True):
    if "vegetable_dataset" in p or "training_results" in p or not os.path.isfile(p): continue
    if not p.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
        if p.lower().endswith((".heic", ".heif")): skipped.append(p + "   <- HEIC: convert to .jpg")
        continue
    name = os.path.basename(p); m = pat.match(name)
    if not m or m.group(1).lower() not in lookup: skipped.append(p); continue
    key = (m.group(1).lower(), m.group(2).lower(), int(m.group(3)))
    if key in seen: skipped.append(p + "   <- duplicate"); continue
    seen.add(key); cls = lookup[m.group(1).lower()]
    shutil.copy(p, f"{OWN}/test/{cls}/{m.group(1).lower()}_{m.group(2).lower()}_{int(m.group(3)):03d}.jpg")
    counts[cls] += 1; people[m.group(2).lower()] += 1
print("Per class:", dict(counts)); print("Per person:", dict(people)); print("Total:", sum(counts.values()))
print("Skipped:", *skipped[:40], sep="\n  ")

Per class: {'Potato': 24, 'Tomato': 24, 'Capsicum': 24, 'Carrot': 24, 'Broccoli': 24}
Per person: {'sumerah': 30, 'alyssa': 30, 'grace': 30, 'antony': 30}
Total: 120
Skipped:
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_001.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_003.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_004.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_002.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_005.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_006.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_alyssa_003.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original pho

## 3. Train the 3 learning-rate runs

In [5]:

from ultralytics import YOLO
import os

for lr in LEARNING_RATES:
    run_name = f"lr_{lr}"

    if os.path.exists(f"{RESULTS}/{run_name}/weights/best.pt"):
        print(f"{run_name} already trained, skipping")
        continue

    print(f"Training with learning rate: {lr}")

    model = YOLO("yolo11n-cls.pt")
    model.train(
        data=DATA,
        batch=16,
        optimizer=OPTIMIZER,
        lr0=lr,
        project=RESULTS,
        name=run_name,
        exist_ok=True,
        **BASE
    )


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Training with learning rate: 0.001
Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/vegetable_dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, form

## 4. Evaluate baseline + all runs on (a) the online test split and (b) our own photos
Performance on the original photographs is emphasized because these images provide an independent evaluation of the models under real-world conditions.

In [6]:

import pandas as pd
from ultralytics import YOLO

runs = {
    "baseline": BASELINE_WEIGHTS,
    **{
        f"lr_{lr}": f"{RESULTS}/lr_{lr}/weights/best.pt"
        for lr in LEARNING_RATES
    }
}

rows = []

for label, weights in runs.items():
    print(f"\nEvaluating {label}")

    model = YOLO(weights)

    online = model.val(
        data=DATA,
        split="test",
        imgsz=224,
        batch=32,
        project=RESULTS,
        name=f"eval_online_{label}",
        exist_ok=True
    )

    own = model.val(
        data=OWN,
        split="test",
        imgsz=224,
        batch=32,
        project=RESULTS,
        name=f"eval_own_{label}",
        exist_ok=True
    )

    rows.append({
        "run": label,
        "online_test_top1": round(online.top1, 4),
        "online_test_top5": round(online.top5, 4),
        "own_photos_top1": round(own.top1, 4),
        "own_photos_top5": round(own.top5, 4)
    })

table = pd.DataFrame(rows)
table.to_csv(
    f"{RESULTS}/learning_rate_comparison.csv",
    index=False
)

display(table)



Evaluating baseline
Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-cls summary (fused): 47 layers, 1,532,429 parameters, 0 gradients, 3.2 GFLOPs
train: /content/vegetable_dataset/train... found 5000 images in 5/5 classes ✅ 
val: /content/vegetable_dataset/val... found 1000 images in 5/5 classes ✅ 
test: /content/vegetable_dataset/test... found 1000 images in 5/5 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1279.7±766.4 MB/s, size: 44.3 KB)
test: Scanning /content/vegetable_dataset/test... 1000 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 4.3Kit/s 0.2s
test: New cache created: /content/vegetable_dataset/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 32/32 11.3it/s 2.8s
                   all          1          1
Speed: 0.1ms preprocess, 0.9ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/drive/MyDrive/IAT 360 CV PROJECT/training_results/eval_online_baseline
Ultral

,run,online_test_top1,online_test_top5,own_photos_top1,own_photos_top5
0,baseline,1.000,1.0,0.7333,1.0
1,lr_0.001,1.000,1.0,0.7750,1.0
2,lr_0.005,1.000,1.0,0.7000,1.0
3,lr_0.02,0.983,1.0,0.5750,1.0


## 5. Per-class accuracy on own photos + list of mistakes (for the report)

In [7]:
from collections import defaultdict
for label, w in runs.items():
    m = YOLO(w); correct = defaultdict(int); total = defaultdict(int); wrong = []
    for c in CLASSES:
        for f in sorted(glob.glob(f"{OWN}/test/{c}/*")):
            r = m.predict(f, imgsz=224, verbose=False)[0]
            pred = r.names[r.probs.top1]; total[c] += 1
            if pred == c: correct[c] += 1
            else: wrong.append((os.path.basename(f), c, pred, round(float(r.probs.top1conf), 2)))
    print(f"\n=== {label} ===")
    print({c: f"{correct[c]}/{total[c]}" for c in CLASSES})
    for x in wrong: print("  WRONG:", x[0], "| true:", x[1], "| predicted:", x[2], "| conf:", x[3])


=== baseline ===
{'Broccoli': '24/24', 'Capsicum': '20/24', 'Carrot': '14/24', 'Potato': '15/24', 'Tomato': '15/24'}
  WRONG: capsicum_antony_003.jpg | true: Capsicum | predicted: Carrot | conf: 0.91
  WRONG: capsicum_antony_004.jpg | true: Capsicum | predicted: Broccoli | conf: 0.55
  WRONG: capsicum_sumerah_005.jpg | true: Capsicum | predicted: Broccoli | conf: 0.85
  WRONG: capsicum_sumerah_006.jpg | true: Capsicum | predicted: Potato | conf: 0.74
  WRONG: carrot_alyssa_001.jpg | true: Carrot | predicted: Tomato | conf: 1.0
  WRONG: carrot_alyssa_002.jpg | true: Carrot | predicted: Broccoli | conf: 0.43
  WRONG: carrot_alyssa_005.jpg | true: Carrot | predicted: Tomato | conf: 0.99
  WRONG: carrot_grace_001.jpg | true: Carrot | predicted: Broccoli | conf: 0.98
  WRONG: carrot_grace_002.jpg | true: Carrot | predicted: Broccoli | conf: 0.99
  WRONG: carrot_grace_003.jpg | true: Carrot | predicted: Broccoli | conf: 0.94
  WRONG: carrot_grace_004.jpg | true: Carrot | predicted: Broccoli

# Saved Experiment Results
All training and evaluation outputs are saved in the shared Google Drive folder:

IAT 360 CV PROJECT/training_results/


# Training Results


*   lr_0.001/ — learning rate of 0.001
*   lr_0.005/ — learning rate of 0.005
* lr_0.02/ — learning rate of 0.02

Each folder contains the trained model weights (best.pt), training metrics (results.csv), and visualizations (results.png).

# Evaluation Results
*   eval_online_lr_*/ — evaluation on the Kaggle test dataset
*   eval_own_lr_*/ — evaluation on the group's 120 original photographs

learning_rate_comparison.csv — comparison of the baseline and all three learning-rate configurations


# Key Finding

The learning rate of 0.001 achieved the highest top-1 accuracy on the original photographs (77.5%), compared with 73.3% for the original baseline. Higher learning rates resulted in lower accuracy on the original photographs.

Note: The original baseline used automatic optimizer selection, while the learning-rate experiments used AdamW.

